In [1]:
import os
import urllib.request
import fitz 
import re


In [2]:
def extract_pdf_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text
file_path = "C:\\Users\\Admin\\Documents\\llm_scratch\\The_Verdict.pdf"
main_text = extract_pdf_text(file_path)
print(len(main_text))  # Print the first 500 characters of the extracted text

21923


In [3]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', main_text)
# preprocessed = [item.strip() for item in preprocessed if item.strip()]
preprocessed_item = []
for item in preprocessed:
    stripped_item = item.strip()
    if stripped_item:
        preprocessed_item.append(stripped_item)
print(preprocessed_item[:30])

['1', 'The', 'Verdict', 'Edith', 'Wharton', '1908', 'Exported', 'from', 'Wikisource', 'on', 'June', '3', ',', '2026', '2', 'I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow']


In [4]:
unique_processed_words = sorted(set(preprocessed_item))
vocab_size = len(unique_processed_words)
print(unique_processed_words[:30])
print(f"Vocabulary size: {vocab_size}")

['!', '"', "'", '(', ')', ',', '--', '.', '0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '1908', '1931', '2', '20', '2026', '21', '22', '3', '4', '5']
Vocabulary size: 1253


In [5]:
vocab = {}
for integer,token in enumerate(unique_processed_words):
    vocab[token] = integer

print(vocab)

{'!': 0, '"': 1, "'": 2, '(': 3, ')': 4, ',': 5, '--': 6, '.': 7, '0': 8, '1': 9, '10': 10, '11': 11, '12': 12, '13': 13, '14': 14, '15': 15, '16': 16, '17': 17, '18': 18, '19': 19, '1908': 20, '1931': 21, '2': 22, '20': 23, '2026': 24, '21': 25, '22': 26, '3': 27, '4': 28, '5': 29, '6': 30, '7': 31, '8': 32, '9': 33, ':': 34, ';': 35, '?': 36, 'A': 37, 'Abigor': 38, 'About': 39, 'AdamBMorgan': 40, 'Ah': 41, 'Among': 42, 'And': 43, 'Are': 44, 'Arrt': 45, 'As': 46, 'At': 47, 'Attribution-ShareAlike': 48, 'AzaToth': 49, 'Be': 50, 'Begin': 51, 'Bender235': 52, 'Blurpeace': 53, 'Boris23': 54, 'Bromskloss': 55, 'Burlington': 56, 'But': 57, 'By': 58, 'Carlo': 59, 'Chicago': 60, 'Claude': 61, 'Come': 62, 'Commons': 63, 'Creative': 64, 'Croft': 65, 'Dbenbenn': 66, 'Destroyed': 67, 'Devonshire': 68, 'Dha': 69, 'Don': 70, 'Dschwen': 71, 'Dubarry': 72, 'During': 73, 'Edith': 74, 'Emperors': 75, 'Exported': 76, 'FDL': 77, 'Florence': 78, 'For': 79, 'GNU': 80, 'Gallery': 81, 'Gideon': 82, 'Gisburn'

In [6]:
for key, value in enumerate(vocab.items()):
    print(value)

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
('0', 8)
('1', 9)
('10', 10)
('11', 11)
('12', 12)
('13', 13)
('14', 14)
('15', 15)
('16', 16)
('17', 17)
('18', 18)
('19', 19)
('1908', 20)
('1931', 21)
('2', 22)
('20', 23)
('2026', 24)
('21', 25)
('22', 26)
('3', 27)
('4', 28)
('5', 29)
('6', 30)
('7', 31)
('8', 32)
('9', 33)
(':', 34)
(';', 35)
('?', 36)
('A', 37)
('Abigor', 38)
('About', 39)
('AdamBMorgan', 40)
('Ah', 41)
('Among', 42)
('And', 43)
('Are', 44)
('Arrt', 45)
('As', 46)
('At', 47)
('Attribution-ShareAlike', 48)
('AzaToth', 49)
('Be', 50)
('Begin', 51)
('Bender235', 52)
('Blurpeace', 53)
('Boris23', 54)
('Bromskloss', 55)
('Burlington', 56)
('But', 57)
('By', 58)
('Carlo', 59)
('Chicago', 60)
('Claude', 61)
('Come', 62)
('Commons', 63)
('Creative', 64)
('Croft', 65)
('Dbenbenn', 66)
('Destroyed', 67)
('Devonshire', 68)
('Dha', 69)
('Don', 70)
('Dschwen', 71)
('Dubarry', 72)
('During', 73)
('Edith', 74)
('Emperors', 75)
('Exported', 76)
('FDL', 77)

In [7]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.vocab = vocab

        self.inv_vocab = {}

        for key, value in vocab.items():
            self.inv_vocab[value] = key


    def tokenize(self, text):
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        result = []

        for token in tokens:
            token = token.strip()

            if token:
                result.append(token)

        return result

    def encode(self, text):
        tokens = self.tokenize(text)

        encoded = []

        for token in tokens:
            encoded.append(self.vocab.get(token, -1))

        return encoded

    def decode(self, token_ids):
        words = []

        for token_id in token_ids:
            words.append(self.inv_vocab.get(token_id, '[UNK]'))

        return ' '.join(words)

In [8]:
tokenizer = SimpleTokenizer(vocab)

text = """Preet Shah"""
ids = tokenizer.encode(text)
print(ids)

decoded_text = tokenizer.decode(ids)
print(decoded_text)

[-1, -1]
[UNK] [UNK]


In [9]:
extended_words = sorted(set(list(preprocessed_item)))
extended_words.extend(["Preet", "Shah"])

extended_vocab = {}
for integer, token in enumerate(extended_words):
    extended_vocab[token] = integer
print(len(extended_vocab))


1255


In [10]:
for ext_key, ext_value in enumerate(list(extended_vocab.items())[-5:]):
    print(ext_value)

('younger', 1250)
('your', 1251)
('yourself', 1252)
('Preet', 1253)
('Shah', 1254)


In [11]:
import re

class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab

        self.int_to_str = {}

        for s, i in vocab.items():
            self.int_to_str[i] = s

    def encode(self, text):

        preprocessed = re.split(
            r'([,.:;?_!"()\']|--|\s)',
            text
        )

        cleaned = []

        for item in preprocessed:
            item = item.strip()

            if item:
                cleaned.append(item)

        preprocessed = []

        for item in cleaned:
            if item in self.str_to_int:
                preprocessed.append(item)
            else:
                preprocessed.append("<|unk|>")

        ids = []

        for s in preprocessed:
            ids.append(self.str_to_int[s])

        return ids

    def decode(self, ids):

        words = []

        for i in ids:
            words.append(self.int_to_str[i])

        text = " ".join(words)

        text = re.sub(
            r'\s+([,.:;?!"()\'])',
            r'\1',
            text
        )

        return text

In [12]:
import importlib
import tiktoken

tokenizer = tiktoken.encoding_for_model("gpt-4o")

In [13]:
text = (
    "_"
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[62]


In [14]:

strings = tokenizer.decode(integers)

print(strings)

_


In [15]:
encoded_text = tokenizer.encode(main_text)
print(encoded_text[:30])
print(main_text[:30])

[16, 198, 976, 181523, 198, 9628, 437, 1656, 121925, 198, 9659, 23, 198, 19946, 295, 591, 60072, 276, 1310, 402, 7843, 220, 18, 11, 220, 1323, 21, 198, 17, 198]
1
The Verdict
Edith Wharton
19


In [16]:
enc_sample = encoded_text[50:2000]
print(len(enc_sample))

1950


In [17]:
context_size = 6
for i in range(0,context_size):
    x = enc_sample[i:i + context_size]
    y = enc_sample[i + 1:i + context_size + 1]

    print(f"x: {x}")
    print(f"y:      {y}")

x: [480, 673, 860, 2212, 19005, 316]
y:      [673, 860, 2212, 19005, 316, 198]
x: [673, 860, 2212, 19005, 316, 198]
y:      [860, 2212, 19005, 316, 198, 1047]
x: [860, 2212, 19005, 316, 198, 1047]
y:      [2212, 19005, 316, 198, 1047, 316]
x: [2212, 19005, 316, 198, 1047, 316]
y:      [19005, 316, 198, 1047, 316, 9598]
x: [19005, 316, 198, 1047, 316, 9598]
y:      [316, 198, 1047, 316, 9598, 484]
x: [316, 198, 1047, 316, 9598, 484]
y:      [198, 1047, 316, 9598, 484, 412]


In [18]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[480] ----> 673
[480, 673] ----> 860
[480, 673, 860] ----> 2212
[480, 673, 860, 2212] ----> 19005
[480, 673, 860, 2212, 19005] ----> 316
[480, 673, 860, 2212, 19005, 316] ----> 198


In [19]:
context_size = 3
for i in range(0,context_size):
    x = enc_sample[i:i + context_size]
    y = enc_sample[i + 1:i + context_size + 1]
    print(f"x: {x}")
    print(f"y:      {y}")
    print(tokenizer.decode(x))
    print("  ",tokenizer.decode(y))

x: [480, 673, 860]
y:      [673, 860, 2212]
 it was no
    was no great
x: [673, 860, 2212]
y:      [860, 2212, 19005]
 was no great
    no great surprise
x: [860, 2212, 19005]
y:      [2212, 19005, 316]
 no great surprise
    great surprise to


In [20]:
import torch

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # print(f"Total number of tokenized inputs: {len(token_ids)}")
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [25]:
# txt = raw text used for training
# batch_size = No of examples used in a batch
# max_length = Length of each input sequence(context sequence)
# stride = Step size for the sliding window
# drop lazt = Whether to drop the last batch if it's smaller than batch_size
# num_workers = Number of subprocesses to use for data loading. 0 means that the data will be loaded in the main process.

def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


In [26]:
len(main_text)

21923

In [28]:
vocab_size = 50257
output_dim = 256
context_length = 1024


token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

batch_size = 8
max_length = 4
dataloader = create_dataloader_v1(
    main_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)

Total number of tokenized inputs: 6175


In [29]:
dataloader = create_dataloader_v1(
    main_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

Total number of tokenized inputs: 6175
[tensor([[  16,  198,  464, 4643]]), tensor([[  198,   464,  4643, 11600]])]


In [32]:
dataloader = create_dataloader_v1(main_text, batch_size=8, max_length=5, stride=2, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Total number of tokenized inputs: 6175
Inputs:
 tensor([[   16,   198,   464,  4643, 11600],
        [  464,  4643, 11600,   198,  7407],
        [11600,   198,  7407,   342,   854],
        [ 7407,   342,   854, 41328,   198],
        [  854, 41328,   198,  1129,  2919],
        [  198,  1129,  2919,   198,  3109],
        [ 2919,   198,  3109,  9213,   422],
        [ 3109,  9213,   422, 11145,   271]])

Targets:
 tensor([[  198,   464,  4643, 11600,   198],
        [ 4643, 11600,   198,  7407,   342],
        [  198,  7407,   342,   854, 41328],
        [  342,   854, 41328,   198,  1129],
        [41328,   198,  1129,  2919,   198],
        [ 1129,  2919,   198,  3109,  9213],
        [  198,  3109,  9213,   422, 11145],
        [ 9213,   422, 11145,   271,  1668]])
